In [ ]:
import kagglehub

path = kagglehub.dataset_download("emmarex/plantdisease")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'plantdisease' dataset.
Path to dataset files: /kaggle/input/plantdisease


In [ ]:
print("Installing PyTorch and torchvision...")
!pip install torch torchvision --quiet
print("PyTorch and torchvision installed successfully.")

print("Installing scikit-learn...")
!pip install scikit-learn --quiet
print("scikit-learn installed successfully.")

Installing PyTorch and torchvision...
PyTorch and torchvision installed successfully.
Installing scikit-learn...
scikit-learn installed successfully.


In [ ]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


print(f"Loading dataset from: {path}")
full_dataset = datasets.ImageFolder(root=path, transform=None)

num_classes = len(full_dataset.classes)
print(f"Found {num_classes} classes: {full_dataset.classes}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = train_transforms
val_dataset.dataset.transform = val_transforms

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print("Data loading and preprocessing complete.")

Loading dataset from: /kaggle/input/plantdisease
Found 2 classes: ['PlantVillage', 'plantvillage']
Training set size: 33020
Validation set size: 8256
Data loading and preprocessing complete.


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import os

def create_mobilenet_model(num_classes):
    model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

    in_features = model.classifier[1].in_features

    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = create_mobilenet_model(num_classes)
model.to(device)

print("MobileNetV2 model created and moved to device:")
print(model)

output_dir = 'models'
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured directory '{output_dir}' exists.")

model_definition_code = """import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

def create_mobilenet_model(num_classes):
    model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model
"""

model_path = os.path.join(output_dir, 'model_v1.py')
with open(model_path, 'w') as f:
    f.write(model_definition_code)

print(f"Model definition saved to {model_path}")

Using device: cuda
MobileNetV2 model created and moved to device:
MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 

In [ ]:
import torch.optim as optim
from sklearn.metrics import accuracy_score

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
print(f"Starting training for {num_epochs} epochs...")

train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)


        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    correct_predictions = 0
    total_predictions = 0
    validation_loss = 0.0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            validation_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_predictions += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    epoch_val_loss = validation_loss / len(val_dataset)
    val_losses.append(epoch_val_loss)

    epoch_val_accuracy = accuracy_score(all_labels, all_predictions)
    val_accuracies.append(epoch_val_accuracy)

    # 5. Print or log the metrics
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {epoch_train_loss:.4f}, "
          f"Val Loss: {epoch_val_loss:.4f}, "
          f"Val Acc: {epoch_val_accuracy:.4f}")

print("Training complete.")

Starting training for 5 epochs...
Epoch 1/5 - Train Loss: 0.6945, Val Loss: 0.6981, Val Acc: 0.5000
Epoch 2/5 - Train Loss: 0.6948, Val Loss: 0.6933, Val Acc: 0.5000
Epoch 3/5 - Train Loss: 0.6948, Val Loss: 0.6966, Val Acc: 0.5000
Epoch 4/5 - Train Loss: 0.6941, Val Loss: 0.6955, Val Acc: 0.5000
Epoch 5/5 - Train Loss: 0.6946, Val Loss: 0.6953, Val Acc: 0.5000
Training complete.


In [ ]:
import torch
import os

model_weights_path = os.path.join('models', 'model_v1.pth')

torch.save(model.state_dict(), model_weights_path)

print(f"Trained model weights saved to {model_weights_path}")

Trained model weights saved to models/model_v1.pth


In [ ]:
import json
import os

results_dir = 'results'
os.makedirs(results_dir, exist_ok=True)
print(f"Ensured directory '{results_dir}' exists.")

metrics_path = os.path.join(results_dir, 'metrics_v1.json')

metrics = {
    'final_val_loss': val_losses[-1],
    'final_val_accuracy': val_accuracies[-1],
    'epochs_trained': num_epochs,
    'train_losses_per_epoch': [f'{l:.4f}' for l in train_losses],
    'val_losses_per_epoch': [f'{l:.4f}' for l in val_losses],
    'val_accuracies_per_epoch': [f'{a:.4f}' for a in val_accuracies]
}

with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=4)

print(f"Evaluation metrics saved to {metrics_path}")
print("Recorded Metrics:")
print(json.dumps(metrics, indent=4))

Ensured directory 'results' exists.
Evaluation metrics saved to results/metrics_v1.json
Recorded Metrics:
{
    "final_val_loss": 0.6953338758890019,
